# V4 vs V5 - people-counting benchmark on the ESP32-S3 data (Kaggle)

V4 = `Nithin031/Vibe-Coded-Pipeline`, branch `v4` (LightGBM + gap-aware CNN, 3-s windows at 20 Hz, 5-window median,
trained on ESP32 data only). V5 = `esp32_counter.py` (causal 30 s baseline, 5-s windows at 50 Hz, 30 s median;
final model also trained on mat80 + WiMANS, 20 s empty-room calibration).
**V4 runs from its own GitHub code**, so its numbers are not re-implemented.

**Same input for both:** the WiCount packets after V4's own CSV reader + timestamp cleaning
(15 one-minute files, ESP32-S3, one meeting room, 0-3 people).

| Benchmark | Split | Why |
|---|---|---|
| **B1 V4 protocol** | V4's 3 folds: counts 1-2 whole files held out, counts 0 and 3 (one file each) cut into 3 time blocks | V4's published setting (exact 0.84 for its LightGBM) - also checks the harness |
| **B2 new session** | leave one whole file out (15 folds) | closer to deployment: the test session was never seen; with only one 0 and one 3 file, an ESP32-only model has never seen those counts |
| **B3 noise** | B1 folds, test files replaced by V4's synthetic noise (7 kinds x 3 levels + "crazy") | robustness; same noisy packets for both |
| **B4 device** | - | size, time per window, time to first answer, what runs on the chip |

**Systems:** V4 LightGBM, V4 gap-CNN, V5 WiCount-only (same data as V4), V5 final (Phase 4 recipe).

**Fair scoring:** each system with its own shipped smoothing; only windows that both can answer in time are scored
(window end >= 15 s, V5's first output), and the first 30 s of the empty file are left out (V5's install calibration).
Metrics: exact, MAE, within 1, recall per count, empty false alarms, 0/1/2/3+ macro F1, whole-file exact,
and under noise the share of windows still answered.

Output: `/kaggle/working/benchmark/` (`BENCHMARK.md`, CSVs, figures).

In [ ]:
%%writefile /kaggle/working/esp32_counter.py
"""ESP32-S3 people counter - reference implementation of the final model.

Everything the firmware has to do, written in Python, packet by packet:

    raw CSI amplitude (52 tones)
      -> per-packet gain normalisation (divide by the median over tones)
      -> 50 Hz bins (packets averaged per 20 ms), gaps up to 60 ms interpolated
      -> causal baseline per tone (EMA, 30 s) -> relative fluctuation z = x / baseline - 1
      -> 5 s windows every 2.5 s (first 10 s skipped while the baseline settles)
      -> 10 count features
      -> log of the magnitude features, minus the empty-room offset (if calibrated)
      -> tree model (LightGBM regression, exported to plain arrays / C header)
      -> clip to the trained count range -> median of the last 30 s -> people count

The batch functions (used for training) and StreamingCounter (what the device runs)
share the same code, and test_esp32_counter.py checks that they give the same numbers.
Nothing here looks at future packets.
"""

import json
import math
import os
import time
from collections import deque

import numpy as np
from scipy.signal import lfilter


# ===============================================================
# Settings (identical on the ESP32)
# ===============================================================
FS = 50                  # sample rate of the shared grid (Hz)
WIN_S = 5.0              # window length (s)
HOP_S = 2.5              # window hop (s)
WIN = int(WIN_S * FS)    # 250 bins
HOP = int(HOP_S * FS)    # 125 bins
N_TONES = 52

MAX_GAP_BINS = 3         # gaps up to 3 bins (60 ms) are interpolated
MAX_MISSING = 0.20       # windows with more missing bins than this are skipped

TAU_S = 30.0             # time constant of the causal per-tone baseline
ALPHA = 1.0 / (TAU_S * FS)
WARMUP_S = 10.0          # no output for the first 10 s (baseline settling)
WARMUP_BINS = int(WARMUP_S * FS)

CAL_S = 20.0             # empty-room calibration after the warm-up
N_CAL_WINDOWS = int((CAL_S - WIN_S) / HOP_S) + 1     # 7 windows

SMOOTH_S = 30.0          # output = median of the last 30 s of window estimates
SMOOTH_N = int(SMOOTH_S / HOP_S)                    # 12 windows

EPS = 1e-6

FEATURE_NAMES = [
    "cv_median",       # median over tones of the fluctuation std
    "cv_p90",          # 90th percentile of the same
    "cv_p10",          # 10th percentile of the same
    "diff_energy",     # mean |x(t) - x(t-1)|  (motion energy)
    "pc1_frac",        # share of variance in the first principal component
    "n_pc_90",         # components needed for 90 % of the variance
    "tone_corr",       # mean correlation between tones
    "low_freq_frac",   # temporal power below 2 Hz (slow body motion)
    "high_freq_frac",  # temporal power above 5 Hz
    "mean_std",        # std of the tone-averaged series
]

# magnitude features go through log() so the empty-room offset acts as a ratio
LOG_FEATURES = ["cv_median", "cv_p90", "cv_p10", "diff_energy", "mean_std"]
LOG_IDX = [FEATURE_NAMES.index(name) for name in LOG_FEATURES]


def settings_dict():
    return {"fs_hz": FS, "win_s": WIN_S, "hop_s": HOP_S, "n_tones": N_TONES,
            "max_gap_bins": MAX_GAP_BINS, "max_missing": MAX_MISSING,
            "tau_s": TAU_S, "warmup_s": WARMUP_S, "cal_s": CAL_S,
            "n_cal_windows": N_CAL_WINDOWS, "smooth_s": SMOOTH_S, "smooth_n": SMOOTH_N,
            "feature_names": FEATURE_NAMES, "log_features": LOG_FEATURES}


# ===============================================================
# Batch preprocessing (training)
# ===============================================================
def gain_normalise(A):
    """Per packet: divide by the median over tones (removes AGC jumps)."""
    return A / (np.median(A, axis=1, keepdims=True) + EPS)


def resample_to_grid(t, A):
    """t: packet times (s, sorted), A: (packets, tones).
    Returns X (bins, tones) with NaN rows for empty bins, and the valid mask."""
    idx = np.floor((t - t[0]) * FS).astype(np.int64)
    n_bins = int(idx[-1]) + 1

    uniq, first = np.unique(idx, return_index=True)        # idx is sorted
    sums = np.add.reduceat(A.astype(np.float64), first, axis=0)
    counts = np.diff(np.append(first, len(idx)))

    X = np.full((n_bins, A.shape[1]), np.nan, dtype=np.float32)
    X[uniq] = sums / counts[:, None]
    valid = np.zeros(n_bins, dtype=bool)
    valid[uniq] = True
    return X, valid


def fill_short_gaps(X, valid):
    """Linear interpolation over gaps of at most MAX_GAP_BINS; longer gaps stay NaN."""
    X, valid = X.copy(), valid.copy()
    missing = np.flatnonzero(~valid)
    if len(missing) == 0:
        return X, valid

    runs = np.split(missing, np.flatnonzero(np.diff(missing) > 1) + 1)
    for run in runs:
        left, right = run[0] - 1, run[-1] + 1
        if len(run) <= MAX_GAP_BINS and left >= 0 and right < len(X):
            w = ((run - left) / (right - left))[:, None]
            X[run] = X[left] * (1 - w) + X[right] * w
            valid[run] = True
    return X, valid


def causal_normalise(X, valid):
    """Each tone relative to its own past (EMA baseline): z = x / baseline - 1.

    The baseline only moves on valid bins and starts at the first valid value.
    Because it is linear, any constant per-tone scale of X cancels out, so the
    Phase 1 series (already divided by the recording mean) give the same z."""
    Z = np.full(X.shape, np.nan, dtype=np.float32)
    rows = np.flatnonzero(valid)
    if len(rows) == 0:
        return Z

    xs = X[rows].astype(np.float64)
    zi = (1.0 - ALPHA) * xs[:1]                             # first output = first value
    baseline, _ = lfilter([ALPHA], [1.0, -(1.0 - ALPHA)], xs, axis=0, zi=zi)

    with np.errstate(invalid="ignore", divide="ignore"):
        z = np.where(baseline > EPS, xs / baseline - 1.0, 0.0)
    Z[rows] = z
    return Z


def window_starts(valid):
    """Window start bins: every HOP, after the warm-up, with enough valid bins."""
    n = len(valid)
    return [a for a in range(0, n - WIN + 1, HOP)
            if a >= WARMUP_BINS and valid[a:a + WIN].mean() >= 1 - MAX_MISSING]


# ===============================================================
# Count features (same code as Phase 1)
# ===============================================================
def window_features(W):
    """W: (bins, 52) normalised window. Missing bins count as 'no change' (0)."""
    W = np.nan_to_num(W.astype(np.float32))
    sd = W.std(axis=0)
    diff_energy = np.abs(np.diff(W, axis=0)).mean()

    Xc = W - W.mean(axis=0)
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    n90 = np.searchsorted(np.cumsum(frac), 0.90) + 1

    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(Xc.T)
    tone_corr = np.nanmean(C[np.triu_indices_from(C, k=1)])

    x = Xc.mean(axis=1)
    P = np.abs(np.fft.rfft(x)) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / FS)
    p_total = P[f > 0].sum() + 1e-12
    low = P[(f > 0) & (f < 2.0)].sum() / p_total
    high = P[f > 5.0].sum() / p_total

    return np.array([np.median(sd), np.percentile(sd, 90), np.percentile(sd, 10),
                     diff_energy, frac[0], n90, tone_corr, low, high, x.std()],
                    dtype=np.float32)


def features_from_series(X, valid):
    """X: (bins, 52) amplitude on the 50 Hz grid (any per-tone scale).
    Returns window start bins and features (windows, 10)."""
    Z = causal_normalise(X, valid)
    starts = window_starts(valid)
    F = np.zeros((len(starts), len(FEATURE_NAMES)), dtype=np.float32)
    for j, a in enumerate(starts):
        F[j] = window_features(Z[a:a + WIN])
    return np.array(starts, dtype=np.int32), F


def features_from_raw(t, A):
    """Raw packets (times s, amplitude (packets, 52)) -> window starts + features."""
    order = np.argsort(t, kind="stable")
    t, A = np.asarray(t, dtype=np.float64)[order], np.asarray(A)[order]
    X, valid = resample_to_grid(t, gain_normalise(A.astype(np.float32)))
    X, valid = fill_short_gaps(X, valid)
    return features_from_series(X, valid)


# ===============================================================
# Model input, calibration, smoothing
# ===============================================================
def prepare_features(F, offset=None):
    """Raw features -> model input: NaN -> 0, log of magnitude features, minus offset."""
    T = np.nan_to_num(np.asarray(F, dtype=np.float64), nan=0.0)
    T = np.atleast_2d(T).copy()
    T[:, LOG_IDX] = np.log(np.maximum(T[:, LOG_IDX], 0.0) + EPS)
    if offset is not None:
        T = T - np.asarray(offset, dtype=np.float64)
    return T.astype(np.float32)


def calibration_offset(F_cal):
    """Empty-room offset = mean model input over the calibration windows."""
    return prepare_features(F_cal).astype(np.float64).mean(axis=0)


def smooth_predictions(pred):
    """Causal rolling median over the last SMOOTH_N window estimates."""
    pred = np.asarray(pred, dtype=np.float64)
    out = np.empty_like(pred)
    for i in range(len(pred)):
        out[i] = np.median(pred[max(0, i - SMOOTH_N + 1):i + 1])
    return out


def to_count(pred, max_count):
    return np.clip(np.rint(pred), 0, max_count).astype(int)


# ===============================================================
# Tree model: LightGBM -> flat arrays (Python, JSON, C header)
# ===============================================================
def _float32_at_most(value):
    """Largest float32 <= value, so 'x <= thr' in float32 equals 'x <= value' in double."""
    thr = np.float32(value)
    if float(thr) > value:
        thr = np.nextafter(thr, np.float32(-np.inf))
    return float(thr)


class TreeModel:
    """Sum of regression trees stored as flat node arrays.

    Node i: feature[i] = -1 for a leaf (value[i] = leaf output), otherwise go to
    left[i] if x[feature[i]] <= value[i] (threshold) else right[i]."""

    def __init__(self, roots, feature, value, left, right, max_count, n_features):
        self.roots = np.asarray(roots, dtype=np.int32)
        self.feature = np.asarray(feature, dtype=np.int32)
        self.value = np.asarray(value, dtype=np.float32)
        self.left = np.asarray(left, dtype=np.int32)
        self.right = np.asarray(right, dtype=np.int32)
        self.max_count = int(max_count)
        self.n_features = int(n_features)

    # ---------------------------------------------------------------
    # Build / save / load
    # ---------------------------------------------------------------
    @classmethod
    def from_lightgbm(cls, booster, max_count):
        dump = booster.dump_model()
        feature, value, left, right, roots = [], [], [], [], []

        def add(node):
            i = len(feature)
            feature.append(-1)
            value.append(0.0)
            left.append(-1)
            right.append(-1)

            if "leaf_value" in node:
                value[i] = float(np.float32(node["leaf_value"]))
                return i

            if node["decision_type"] != "<=":
                raise ValueError(f"unsupported split type {node['decision_type']}")
            if node.get("missing_type", "None") not in ("None", "NaN"):
                raise ValueError(f"unsupported missing type {node['missing_type']}")

            feature[i] = int(node["split_feature"])
            value[i] = _float32_at_most(float(node["threshold"]))
            left[i] = add(node["left_child"])
            right[i] = add(node["right_child"])
            return i

        for tree in dump["tree_info"]:
            roots.append(add(tree["tree_structure"]))

        return cls(roots, feature, value, left, right, max_count,
                   n_features=dump["max_feature_idx"] + 1)

    def to_dict(self):
        return {"roots": self.roots.tolist(), "feature": self.feature.tolist(),
                "value": [float(v) for v in self.value], "left": self.left.tolist(),
                "right": self.right.tolist(), "max_count": self.max_count,
                "n_features": self.n_features}

    def save(self, path):
        with open(path, "w") as f:
            json.dump(self.to_dict(), f)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            d = json.load(f)
        return cls(d["roots"], d["feature"], d["value"], d["left"], d["right"],
                   d["max_count"], d["n_features"])

    # ---------------------------------------------------------------
    # Inference (float32, same order of operations as the C code)
    # ---------------------------------------------------------------
    def predict(self, X):
        X = np.atleast_2d(np.asarray(X, dtype=np.float32))
        if np.isnan(X).any():
            raise ValueError("model input contains NaN - use prepare_features()")

        rows = np.arange(len(X))
        total = np.zeros(len(X), dtype=np.float32)
        for root in self.roots:
            node = np.full(len(X), root, dtype=np.int32)
            while True:
                f = self.feature[node]
                split = f >= 0
                if not split.any():
                    break
                go_left = X[rows, np.maximum(f, 0)] <= self.value[node]
                node = np.where(split, np.where(go_left, self.left[node], self.right[node]), node)
            total += self.value[node]
        return total

    @property
    def n_nodes(self):
        return len(self.feature)

    # ---------------------------------------------------------------
    # C header for the ESP32 firmware
    # ---------------------------------------------------------------
    def write_c_header(self, path, note=""):
        idx_type = "int16_t" if self.n_nodes < 32767 else "int32_t"

        def array(ctype, name, values, fmt):
            body = ", ".join(fmt(v) for v in values)
            lines, line = [], "    "
            for item in body.split(", "):
                if len(line) + len(item) > 96:
                    lines.append(line.rstrip())
                    line = "    "
                line += item + ", "
            lines.append(line.rstrip().rstrip(","))
            return f"static const {ctype} {name}[{len(values)}] = {{\n" + "\n".join(lines) + "\n};\n"

        log_flags = [1 if name in LOG_FEATURES else 0 for name in FEATURE_NAMES]
        float_fmt = lambda v: f"{float(v):.9g}f"
        int_fmt = lambda v: str(int(v))

        text = [
            "/* ESP32-S3 people counter - generated by esp32_counter.py. Do not edit. */",
            f"/* {note} */" if note else "",
            "#ifndef CC_COUNTER_MODEL_H",
            "#define CC_COUNTER_MODEL_H",
            "",
            "#include <math.h>",
            "#include <stdint.h>",
            "",
            "/* ---- preprocessing settings (must match esp32_counter.py) ---- */",
            f"#define CC_FS_HZ          {FS}",
            f"#define CC_N_TONES        {N_TONES}",
            f"#define CC_WIN_BINS       {WIN}",
            f"#define CC_HOP_BINS       {HOP}",
            f"#define CC_MAX_GAP_BINS   {MAX_GAP_BINS}",
            f"#define CC_MAX_MISSING    {MAX_MISSING}f",
            f"#define CC_EMA_ALPHA      {ALPHA:.9g}f",
            f"#define CC_WARMUP_BINS    {WARMUP_BINS}",
            f"#define CC_N_CAL_WINDOWS  {N_CAL_WINDOWS}",
            f"#define CC_SMOOTH_N       {SMOOTH_N}",
            "",
            "/* ---- model ---- */",
            f"#define CC_N_FEATURES     {self.n_features}",
            f"#define CC_N_TREES        {len(self.roots)}",
            f"#define CC_N_NODES        {self.n_nodes}",
            f"#define CC_MAX_COUNT      {self.max_count}",
            "",
            "/* features, in order: " + ", ".join(FEATURE_NAMES) + " */",
            array("uint8_t", "cc_log_feature", log_flags, int_fmt),
            array(idx_type, "cc_roots", self.roots, int_fmt),
            array("int8_t", "cc_feature", self.feature, int_fmt),
            array("float", "cc_value", self.value, float_fmt),
            array(idx_type, "cc_left", self.left, int_fmt),
            array(idx_type, "cc_right", self.right, int_fmt),
            "/* raw window features -> model input. offset = NULL when not calibrated. */",
            "static inline void cc_prepare(const float *raw, const float *offset, float *out)",
            "{",
            "    for (int i = 0; i < CC_N_FEATURES; i++) {",
            "        float v = isnan(raw[i]) ? 0.0f : raw[i];",
            "        if (cc_log_feature[i]) {",
            "            v = logf(fmaxf(v, 0.0f) + 1e-6f);",
            "        }",
            "        if (offset) {",
            "            v -= offset[i];",
            "        }",
            "        out[i] = v;",
            "    }",
            "}",
            "",
            "/* model input -> people estimate (not yet clipped / smoothed) */",
            "static inline float cc_predict(const float *x)",
            "{",
            "    float total = 0.0f;",
            "    for (int t = 0; t < CC_N_TREES; t++) {",
            "        int n = cc_roots[t];",
            "        while (cc_feature[n] >= 0) {",
            "            n = (x[cc_feature[n]] <= cc_value[n]) ? cc_left[n] : cc_right[n];",
            "        }",
            "        total += cc_value[n];",
            "    }",
            "    return total;",
            "}",
            "",
            "#endif /* CC_COUNTER_MODEL_H */",
            "",
        ]
        with open(path, "w") as f:
            f.write("\n".join(text))
        return os.path.getsize(path)

    def flash_bytes(self):
        """Approximate flash used by the node arrays in the C header."""
        idx_bytes = 2 if self.n_nodes < 32767 else 4
        per_node = 1 + 4 + 2 * idx_bytes                    # feature + value + left + right
        return self.n_nodes * per_node + len(self.roots) * idx_bytes


# ===============================================================
# Streaming counter (what the ESP32 runs)
# ===============================================================
class StreamingCounter:
    """Feed packets one by one with push(t_seconds, amplitude_52).

    Every 2.5 s (after the warm-up) push() returns a result dict with the window
    features and, if a model is loaded, the raw / smoothed estimate and the count.
    Packets must arrive in time order; late packets are dropped."""

    def __init__(self, model=None, offset=None):
        self.model = model
        self.offset = None if offset is None else np.asarray(offset, dtype=np.float64)

        # 50 Hz binning
        self.t0 = None
        self.cur_bin = -1
        self.acc_sum = None
        self.acc_n = 0
        self.last_valid_bin = -1
        self.last_valid_x = None

        # causal baseline + window buffer
        self.baseline = None
        self.z_buf = deque(maxlen=WIN)
        self.v_buf = deque(maxlen=WIN)
        self.n_bins = 0

        # output smoothing + calibration
        self.recent = deque(maxlen=SMOOTH_N)
        self.cal_feats = None
        self.cal_target = 0

    # ---------------------------------------------------------------
    # Calibration: leave the room empty, call begin_calibration(), keep pushing
    # ---------------------------------------------------------------
    def begin_calibration(self, n_windows=N_CAL_WINDOWS):
        self.cal_feats = []
        self.cal_target = n_windows

    @property
    def calibrating(self):
        return self.cal_feats is not None

    # ---------------------------------------------------------------
    # Packet input
    # ---------------------------------------------------------------
    def push(self, t, amplitude):
        x = gain_normalise(np.asarray(amplitude, dtype=np.float32)[None])[0]
        if self.t0 is None:
            self.t0 = float(t)
        b = int(math.floor((float(t) - self.t0) * FS))

        results = []
        if b < self.cur_bin:
            return results                                  # late packet

        if b > self.cur_bin:
            if self.acc_n:
                results += self._close_bin(self.cur_bin, self.acc_sum / self.acc_n)
            self.cur_bin = b
            self.acc_sum = np.zeros(N_TONES, dtype=np.float64)
            self.acc_n = 0

        self.acc_sum += x
        self.acc_n += 1
        return results

    def flush(self):
        """Close the last open bin (end of a recording)."""
        results = []
        if self.acc_n:
            results = self._close_bin(self.cur_bin, self.acc_sum / self.acc_n)
            self.acc_n = 0
        return results

    # ---------------------------------------------------------------
    # Internals
    # ---------------------------------------------------------------
    def _close_bin(self, k, mean):
        x = mean.astype(np.float32)
        out = []

        if self.last_valid_bin >= 0:
            gap = k - self.last_valid_bin - 1
            if 0 < gap <= MAX_GAP_BINS:
                left = self.last_valid_bin
                for j in range(left + 1, k):
                    w = (j - left) / (k - left)
                    xi = (self.last_valid_x * (1 - w) + x * w).astype(np.float32)
                    out += self._add_bin(xi, True)
            else:
                for _ in range(gap):
                    out += self._add_bin(None, False)

        self.last_valid_bin, self.last_valid_x = k, x
        out += self._add_bin(x, True)
        return out

    def _add_bin(self, x, valid):
        if valid:
            xd = x.astype(np.float64)
            if self.baseline is None:
                self.baseline = xd.copy()
            else:
                self.baseline = self.baseline + ALPHA * (xd - self.baseline)
            with np.errstate(invalid="ignore", divide="ignore"):
                z = np.where(self.baseline > EPS, xd / self.baseline - 1.0, 0.0)
            z = z.astype(np.float32)
        else:
            z = np.full(N_TONES, np.nan, dtype=np.float32)

        self.z_buf.append(z)
        self.v_buf.append(valid)
        self.n_bins += 1

        a = self.n_bins - WIN
        if a < 0 or a % HOP or a < WARMUP_BINS:
            return []
        if np.mean(self.v_buf) < 1 - MAX_MISSING:
            return []

        feats = window_features(np.array(self.z_buf))
        return [self._decide(a, feats)]

    def _decide(self, start_bin, feats):
        res = {"t_start_s": start_bin / FS, "t_end_s": (start_bin + WIN) / FS,
               "features": feats}

        if self.calibrating:
            self.cal_feats.append(feats)
            res["status"] = "calibrating"
            if len(self.cal_feats) >= self.cal_target:
                self.offset = calibration_offset(np.array(self.cal_feats))
                self.cal_feats = None
                self.recent.clear()
                res["status"] = "calibrated"
            return res

        if self.model is None:
            return res

        x = prepare_features(feats[None], self.offset)
        raw = float(np.clip(self.model.predict(x)[0], 0, self.model.max_count))
        self.recent.append(raw)
        smoothed = float(np.median(self.recent))
        res.update(status="ok", raw=raw, smoothed=smoothed,
                   count=int(to_count(smoothed, self.model.max_count)))
        return res


# ===============================================================
# Model bundle (what goes to the device / the friend's demo)
# ===============================================================
def save_bundle(model_dir, tree_model, card, booster=None):
    """Writes trees.json, model_card.json, counter_model.h (+ model.txt if booster)."""
    os.makedirs(model_dir, exist_ok=True)
    tree_model.save(os.path.join(model_dir, "trees.json"))

    note = f"{card.get('name', 'counter')} | calibration={card.get('calibration')} | " \
           f"counts 0-{tree_model.max_count} | {time.strftime('%Y-%m-%d %H:%M')}"
    header_bytes = tree_model.write_c_header(os.path.join(model_dir, "counter_model.h"), note)

    if booster is not None:
        booster.save_model(os.path.join(model_dir, "model.txt"))

    card = {**card, "settings": settings_dict(), "n_trees": len(tree_model.roots),
            "n_nodes": tree_model.n_nodes, "flash_bytes_est": tree_model.flash_bytes(),
            "header_file_bytes": header_bytes}
    with open(os.path.join(model_dir, "model_card.json"), "w") as f:
        json.dump(card, f, indent=1)
    return card


def load_bundle(model_dir):
    """Returns (TreeModel, card). Use with StreamingCounter(model)."""
    model = TreeModel.load(os.path.join(model_dir, "trees.json"))
    with open(os.path.join(model_dir, "model_card.json")) as f:
        card = json.load(f)
    if card["settings"] != settings_dict():
        raise ValueError("model_card settings differ from this esp32_counter.py")
    return model, card

In [ ]:
# 1. Setup: V4 from GitHub, WiCount raw data, V5 code
import glob
import json
import os
import subprocess
import sys
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Locate dataset folders under /kaggle/input (works for any dataset slug / nesting)
SKIP_DIRS = {"amp", "mat", "video"}            # huge leaf folders: never walk into them


def find_dirs(test, root="/kaggle/input", max_depth=6):
    """Return folders for which test(folder, file_names) is True (shallow walk)."""
    hits = []
    base = root.rstrip("/").count("/")
    for folder, subdirs, files in os.walk(root):
        if test(folder, files):
            hits.append(folder)
        if folder.count("/") - base >= max_depth:
            subdirs[:] = []
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS and not s.startswith("sub_")]
    return sorted(hits)


print("attached datasets:", sorted(os.listdir("/kaggle/input")))

REPO = "/kaggle/working/vcp"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "v4",
                    "https://github.com/Nithin031/Vibe-Coded-Pipeline.git", REPO], check=True)
V4_COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("V4 commit:", V4_COMMIT)

hits = find_dirs(lambda d, f: os.path.basename(d) == "data" and "WiCount" in d
                 and os.path.isdir(os.path.join(d, "0")), max_depth=8)
assert hits, "WiCount/data not found - attach wizard1803/wifi-csi"
WICOUNT_DATA = hits[0]

# V4 reads <CSI_DATASETS>/wicount/extracted/data/<count>/*.csv
V4DATA = "/kaggle/working/v4data"
os.makedirs(os.path.join(V4DATA, "wicount", "extracted"), exist_ok=True)
link = os.path.join(V4DATA, "wicount", "extracted", "data")
if not os.path.exists(link):
    os.symlink(WICOUNT_DATA, link)
os.environ["CSI_DATASETS"] = V4DATA

sys.path.insert(0, os.path.join(REPO, "V4", "src"))
import data as v4data
import metrics as v4metrics
import models as v4models
import noise as v4noise
import run_counting as v4count

sys.path.insert(0, "/kaggle/working")
import esp32_counter as ec

OUT = "/kaggle/working/benchmark"
os.makedirs(os.path.join(OUT, "figures"), exist_ok=True)
SEED = 0
V5_PARAMS = dict(objective="regression", n_estimators=300, learning_rate=0.05, num_leaves=15,
                 min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                 random_state=SEED, verbose=-1)
FIRST_END_S = ec.WARMUP_S + ec.WIN_S           # 15 s: V5's first answer
CAL_END_S = ec.WARMUP_S + ec.CAL_S             # 30 s: end of V5's install calibration
print("WiCount:", WICOUNT_DATA)

In [ ]:
# 2. The same WiCount packets for both: V4 windows (its code) and V5 windows (esp32_counter.py)
wc, wseq, wgap = v4data.load(["wicount"], "standard")
RECS = list(v4data.recordings("wicount"))                    # V4 order (noise seeds follow it)
V4_COLS = v4data.feature_cols(wc)


def v5_windows(rec):
    """V5 device pipeline on one V4 Recording (52 amplitudes per packet)."""
    starts, F = ec.features_from_raw(rec.t, rec.amp)
    d = pd.DataFrame(F, columns=ec.FEATURE_NAMES)
    d["recording"] = rec.meta["recording"]
    d["count"] = int(rec.meta["count"])
    d["window"] = np.arange(len(d))
    d["t_start_s"] = starts / ec.FS
    return d


v5 = pd.concat([v5_windows(r) for r in RECS], ignore_index=True)
print("V4 windows:", len(wc), f"(valid {wc['valid'].mean():.1%}) | V5 windows:", len(v5))
display(pd.DataFrame({"V4": wc.groupby("count")["recording"].nunique(),
                      "V5": v5.groupby("count")["recording"].nunique()}).rename_axis("count"))


# ---- V5 final model also trains on mat80 + WiMANS (Phase 5a cues, base 10 only) ----
def find_input(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found - attach the wifi-phase5-features output"
    return hits[0]


st = pd.read_parquet(find_input("streams_features.parquet"))
wim = pd.read_parquet(find_input("wimans_clip_features.parquet"))
mat = st[st["dataset"] == "mat80"].copy()
wim["window"], wim["t_start_s"] = 0, 0.0

# install calibration of the extra rooms (as in Phase 4): never trained on
mat["is_cal"] = False
for setup, g in mat[mat["count"] == 0].groupby("setup"):
    rec = sorted(g["recording"].unique())[0]
    mat.loc[(mat["recording"] == rec) & (mat["t_start_s"] + ec.WIN_S <= CAL_END_S + 1e-9), "is_cal"] = True
wim["is_cal"] = False
for setup, g in wim[wim["count"] == 0].groupby("setup"):
    first = sorted(g["session"].unique())[0]
    clips = (g[g["session"] == first].drop_duplicates("recording").sort_values("sample_no")["recording"]
             .head(ec.N_CAL_WINDOWS))
    wim.loc[wim["recording"].isin(clips), "is_cal"] = True

EXTRA_ROWS, EXTRA_OFFSETS = [], {}
for ds, d in [("mat80", mat), ("wimans", wim)]:
    for (setup, view), g in d[d["is_cal"]].groupby(["setup", "view"]):
        EXTRA_OFFSETS[(ds, setup, view)] = ec.calibration_offset(g[ec.FEATURE_NAMES].values)
    rows = d[~d["is_cal"]].assign(dataset=ds)
    EXTRA_ROWS.append(rows[["dataset", "setup", "view", "count"] + ec.FEATURE_NAMES])
EXTRA_ROWS = pd.concat(EXTRA_ROWS, ignore_index=True)
print("V5 extra training rows:", EXTRA_ROWS["dataset"].value_counts().to_dict())

In [ ]:
# 3. Splits: V4's 3 folds (applied identically to V5's windows) and leave-one-file-out
N_FOLDS = 3
SINGLE = {c for c, g in wc.groupby("count") if g["recording"].nunique() < N_FOLDS}   # counts 0 and 3
BLOCK_END = {rec: g["t0"].max() + 3.0 for rec, g in wc.groupby("recording")}       # V4's block edges


def v4_fold_masks(table, k):
    return v4count.fold_masks(table, k)


def v5_fold_masks(table, k):
    """V4's rule on V5 windows: same held-out files; same time blocks for the single files."""
    train = np.zeros(len(table), bool)
    test = np.zeros(len(table), bool)
    _, v4_test = v4count.fold_masks(wc, k)
    test_recs = set(wc.loc[v4_test & ~wc["count"].isin(SINGLE), "recording"])
    for rec, g in table.groupby("recording"):
        idx = g.index.to_numpy()
        if g["count"].iloc[0] in SINGLE:
            edges = np.linspace(0, BLOCK_END[rec], N_FOLDS + 1)
            t = g["t_start_s"].to_numpy()
            for b in range(N_FOLDS):
                inside = (t >= edges[b]) & (t + ec.WIN_S <= edges[b + 1])
                (test if b == k else train)[idx[inside]] = True
        elif rec in test_recs:
            test[idx] = True
        else:
            train[idx] = True
    return train, test


def lofo_masks(table, rec):
    test = (table["recording"] == rec).to_numpy()
    return ~test, test


PROTOCOLS = {
    "B1_v4_folds": [(f"fold{k}", (lambda t, k=k: v4_fold_masks(t, k)), (lambda t, k=k: v5_fold_masks(t, k)))
                    for k in range(N_FOLDS)],
    "B2_new_session": [(rec, (lambda t, r=rec: lofo_masks(t, r)), (lambda t, r=rec: lofo_masks(t, r)))
                       for rec in sorted(wc["recording"].unique())],
}
for name, folds in PROTOCOLS.items():
    print(name, len(folds), "folds")

In [ ]:
# 4. The four systems: train on a fold, return a predictor (window table -> count estimate per window)
def train_v4(kind, train_mask):
    mask = train_mask & wc["valid"].to_numpy()
    predict, _, _ = v4count.fit_predict(kind, wc, wseq, wgap, mask, f"bench-{kind}")
    return lambda table, seq, gap: predict(table, seq, gap).argmax(1).astype(float)


def v5_offset(table):
    """Install calibration: first 20 s after the warm-up of the empty file."""
    empty = table[(table["count"] == 0) & (table["t_start_s"] + ec.WIN_S <= CAL_END_S + 1e-9)]
    return ec.calibration_offset(empty[ec.FEATURE_NAMES].values) if len(empty) else None


def train_v5(train_table, final):
    """final=False: WiCount only, no calibration. final=True: + mat80 + WiMANS, empty-room calibration."""
    wic = train_table[~((train_table["count"] == 0) & (train_table["t_start_s"] + ec.WIN_S <= CAL_END_S + 1e-9))]
    if not final:
        X, y, w = ec.prepare_features(wic[ec.FEATURE_NAMES].values), wic["count"].values, np.ones(len(wic))
    else:
        off = v5_offset(v5)                                    # the room's install calibration
        parts_X = [ec.prepare_features(wic[ec.FEATURE_NAMES].values, off)]
        parts_y, parts_d = [wic["count"].values], [np.full(len(wic), "wicount")]
        for key, idx in EXTRA_ROWS.groupby(["dataset", "setup", "view"]).indices.items():
            if key in EXTRA_OFFSETS:
                g = EXTRA_ROWS.iloc[idx]
                parts_X.append(ec.prepare_features(g[ec.FEATURE_NAMES].values, EXTRA_OFFSETS[key]))
                parts_y.append(g["count"].values)
                parts_d.append(g["dataset"].values)
        X, y, d = np.concatenate(parts_X), np.concatenate(parts_y), np.concatenate(parts_d)
        w = 1.0 / pd.Series(d).map(pd.Series(d).value_counts()).values     # each dataset the same weight
    reg = lgb.LGBMRegressor(**V5_PARAMS).fit(X, y, sample_weight=w / w.mean())
    max_count = int(y.max())

    def predict(table, offset=None):
        off = (offset if offset is not None else v5_offset(v5)) if final else None
        return np.clip(reg.predict(ec.prepare_features(table[ec.FEATURE_NAMES].values, off)), 0, max_count)
    return predict, reg, max_count


SYSTEM_NAMES = ["V4 LightGBM", "V4 gap-CNN", "V5 WiCount-only", "V5 final"]

In [ ]:
# 5. One scoring for everybody: own smoothing, common time span, same metrics
def smooth_v4(table, pred):
    return v4metrics.smooth(table.reset_index(drop=True), pred)          # 5-window causal median (V4)


def smooth_v5(table, pred):
    d = table[["recording", "window"]].reset_index(drop=True).assign(p=pred)
    return d.groupby("recording")["p"].transform(lambda s: ec.smooth_predictions(s.values)).values


def frame(system, table, pred, t_start, win_s, smoothed):
    return pd.DataFrame({"system": system, "recording": table["recording"].values, "count": table["count"].values,
                         "t_end_s": t_start + win_s, "raw": pred, "smoothed": smoothed})


def common_span(f):
    keep = f["t_end_s"] >= FIRST_END_S - 1e-9
    keep &= ~((f["count"] == 0) & (f["t_end_s"] <= CAL_END_S + 1e-9))
    return f[keep]


def f1_macro(y, r):
    out = []
    for c in sorted(set(y) | set(r)):
        tp = np.sum((y == c) & (r == c))
        out.append(2 * tp / max(2 * tp + np.sum((y != c) & (r == c)) + np.sum((y == c) & (r != c)), 1))
    return float(np.mean(out))


def score(f):
    y = f["count"].values.astype(int)
    r = np.clip(np.rint(f["smoothed"].values), 0, 10).astype(int)
    out = {"windows": len(y), "exact": float(np.mean(r == y)), "mae": float(np.mean(np.abs(r - y))),
           "within1": float(np.mean(np.abs(r - y) <= 1)),
           "f1_4": f1_macro(np.minimum(y, 3), np.minimum(r, 3)),
           "empty_false_alarm": float(np.mean(r[y == 0] > 0)) if (y == 0).any() else np.nan}
    recalls = []
    for c in range(4):
        m = y == c
        out[f"recall_{c}"] = float(np.mean(r[m] == c)) if m.any() else np.nan
        if m.any():
            recalls.append(out[f"recall_{c}"])
    out["balanced_acc"] = float(np.mean(recalls))
    per_file = f.groupby("recording").agg(y=("count", "first"), p=("smoothed", "median"))
    out["file_exact"] = float(np.mean(np.rint(per_file["p"]) == per_file["y"]))
    return out

In [ ]:
# 6. B1 + B2: train every system on every fold, predict the held-out windows
FRAMES, FOLD_MODELS = [], {}
t0 = time.time()
for proto, folds in PROTOCOLS.items():
    for fold_name, m4, m5 in folds:
        tr4, te4 = m4(wc)
        tr5, te5 = m5(v5)
        test4 = wc[te4 & wc["valid"].to_numpy()].reset_index(drop=True)
        seq4, gap4 = wseq[te4 & wc["valid"].to_numpy()], wgap[te4 & wc["valid"].to_numpy()]
        test5 = v5[te5].sort_values(["recording", "window"]).reset_index(drop=True)

        models = {"V4 LightGBM": train_v4("lgbm", tr4), "V4 gap-CNN": train_v4("gapcnn", tr4)}
        for name, fn in models.items():
            p = fn(test4, seq4, gap4)
            FRAMES.append(frame(name, test4, p, test4["t0"].values, 3.0, smooth_v4(test4, p))
                          .assign(protocol=proto, fold=fold_name))

        v5_models = {"V5 WiCount-only": train_v5(v5[tr5], final=False),
                     "V5 final": train_v5(v5[tr5], final=True)}
        for name, (fn, _, _) in v5_models.items():
            p = fn(test5)
            FRAMES.append(frame(name, test5, p, test5["t_start_s"].values, ec.WIN_S, smooth_v5(test5, p))
                          .assign(protocol=proto, fold=fold_name))

        if proto == "B1_v4_folds":
            FOLD_MODELS[fold_name] = {**models, **{k: v[0] for k, v in v5_models.items()}}
        print(f"{proto:15s} {fold_name:32s} [{time.time() - t0:.0f} s]", flush=True)

ALL = pd.concat(FRAMES, ignore_index=True)

# ---- harness check: V4 LightGBM on V4's own protocol and windows (published pooled exact 0.84) ----
v4_check = ALL[(ALL["protocol"] == "B1_v4_folds") & (ALL["system"] == "V4 LightGBM")]
print(f"harness check - V4 LightGBM, B1, all V4 windows: exact {score(v4_check)['exact']:.3f} (published 0.84)")

rows = []
for (proto, system), f in ALL.groupby(["protocol", "system"]):
    rows.append({"protocol": proto, "system": system, **score(common_span(f))})
MAIN = pd.DataFrame(rows)
MAIN["system"] = pd.Categorical(MAIN["system"], SYSTEM_NAMES, ordered=True)
MAIN = MAIN.sort_values(["protocol", "system"])
MAIN.round(3)

In [ ]:
# 7. B3: V4's noise suite on the B1 test windows (same noisy packets for V4 and V5)
CONDS = [(k, lv) for k in v4noise.KINDS for lv in ([3] if k == "crazy" else [1, 2, 3])]
clean_answered = ALL[ALL["protocol"] == "B1_v4_folds"].pipe(common_span).groupby("system").size()
noise_rows = []
t0 = time.time()
for kind, level in CONDS:
    noisy = [v4noise.apply(rec, kind, level, seed=2000 + i) for i, rec in enumerate(RECS)]

    # V4 windows of the noisy stream (its code), aligned to the clean windows by window_id
    rows4, sq, gp = [], [], []
    for rec in noisy:
        r, q, g = v4data.make_windows(rec)
        rows4 += r
        sq.append(q)
        gp.append(g)
    nt, ns, ng = pd.DataFrame(rows4), np.concatenate(sq), np.concatenate(gp)
    nv5 = pd.concat([v5_windows(rec) for rec in noisy if len(rec.t) > ec.FS], ignore_index=True)
    noisy_offset = v5_offset(nv5)                              # calibration done in the same conditions

    frames = []
    for k in range(N_FOLDS):
        fold = f"fold{k}"
        _, te4 = v4_fold_masks(wc, k)
        pos = v4count.align(wc[te4], nt)
        sub = nt.iloc[pos[pos >= 0]].reset_index(drop=True)
        keep = sub["valid"].to_numpy()                          # V4 answers 'unknown' on bad windows
        sub, s_seq, s_gap = sub[keep].reset_index(drop=True), ns[pos[pos >= 0]][keep], ng[pos[pos >= 0]][keep]
        for name in ["V4 LightGBM", "V4 gap-CNN"]:
            p = FOLD_MODELS[fold][name](sub, s_seq, s_gap)
            frames.append(frame(name, sub, p, sub["t0"].values, 3.0, smooth_v4(sub, p)))

        _, te5 = v5_fold_masks(nv5, k)
        sub5 = nv5[te5].sort_values(["recording", "window"]).reset_index(drop=True)
        for name in ["V5 WiCount-only", "V5 final"]:
            fn = FOLD_MODELS[fold][name]
            p = fn(sub5, noisy_offset) if name == "V5 final" else fn(sub5)
            frames.append(frame(name, sub5, p, sub5["t_start_s"].values, ec.WIN_S, smooth_v5(sub5, p)))

    for system, f in pd.concat(frames, ignore_index=True).groupby("system"):
        f = common_span(f)
        noise_rows.append({"noise": kind, "level": level, "system": system,
                           "answered": len(f) / clean_answered[system], **score(f)})
    print(f"{kind:17s} level {level}  [{time.time() - t0:.0f} s]", flush=True)

NOISE = pd.DataFrame(noise_rows)
NOISE["system"] = pd.Categorical(NOISE["system"], SYSTEM_NAMES, ordered=True)
noise_summary = NOISE.groupby("system", observed=True)[["exact", "mae", "f1_4", "answered"]].mean()
display(noise_summary.round(3))
NOISE.pivot_table(index=["noise", "level"], columns="system", values="exact", observed=True).round(3)

In [ ]:
# 8. B4: device side - size, time per window, time to first answer, what runs on the ESP32
# final models on all WiCount (+ extra rooms for V5 final)
all4 = np.ones(len(wc), bool)
v4_lgbm_pred, _, v4_lgbm = v4count.fit_predict("lgbm", wc, wseq, wgap, all4 & wc["valid"].to_numpy(), "all")
_, _, v4_cnn = v4count.fit_predict("gapcnn", wc, wseq, wgap, all4 & wc["valid"].to_numpy(), "all")
_, v5_reg, v5_max = train_v5(v5, final=True)
v5_tree = ec.TreeModel.from_lightgbm(v5_reg.booster_, v5_max)


def lgbm_nodes(booster):
    return sum(2 * t["num_leaves"] - 1 for t in booster.dump_model()["tree_info"])


v4_nodes = sum(lgbm_nodes(m.booster_) for m in v4_lgbm)
v4_cnn_params = sum(p.numel() for p in v4_cnn.parameters())

# time per window on this CPU (preprocessing + features + model), batch size 1
rec = RECS[1]
t0 = time.perf_counter()
rows4, q4, g4 = v4data.make_windows(rec)
v4_feat_ms = (time.perf_counter() - t0) / len(rows4) * 1000
t4 = pd.DataFrame(rows4)
t0 = time.perf_counter()
for i in range(50):
    v4_lgbm_pred(t4.iloc[i % len(t4):i % len(t4) + 1], q4[i % len(t4):i % len(t4) + 1], g4[i % len(t4):i % len(t4) + 1])
v4_lgbm_ms = (time.perf_counter() - t0) / 50 * 1000

t0 = time.perf_counter()
s5, f5 = ec.features_from_raw(rec.t, rec.amp)
v5_feat_ms = (time.perf_counter() - t0) / len(s5) * 1000
x5 = ec.prepare_features(f5)
t0 = time.perf_counter()
for i in range(50):
    v5_tree.predict(x5[i % len(x5):i % len(x5) + 1])
v5_tree_ms = (time.perf_counter() - t0) / 50 * 1000

DEV = pd.DataFrame([
    {"item": "input rate / window / hop", "V4": "20 Hz / 3 s / 1 s", "V5": "50 Hz / 5 s / 2.5 s"},
    {"item": "smoothing (display)", "V4": "median of 5 windows (~5 s)", "V5": "median of 12 windows (30 s)"},
    {"item": "first answer after start", "V4": "~3 s", "V5": f"{FIRST_END_S:.0f} s (10 s warm-up + 5 s window)"},
    {"item": "install step", "V4": "none for counting", "V5": "20 s empty room"},
    {"item": "counting model size", "V4": f"LightGBM 3 seeds x 6 classes: {v4_nodes} nodes "
                                          f"(~{v4_nodes * 11 / 1024:.0f} KB as arrays); gap-CNN {v4_cnn_params} params "
                                          f"(~{v4_cnn_params * 4 / 1024:.0f} KB fp32)",
     "V5": f"{len(v5_tree.roots)} trees, {v5_tree.n_nodes} nodes (~{v5_tree.flash_bytes() / 1024:.0f} KB)"},
    {"item": "CPU time per window (this Kaggle CPU, Python)",
     "V4": f"features {v4_feat_ms:.2f} ms + LightGBM {v4_lgbm_ms:.2f} ms",
     "V5": f"features {v5_feat_ms:.2f} ms + trees {v5_tree_ms:.2f} ms"},
    {"item": "on-chip code", "V4": "none (Python + PyTorch)", "V5": "C header with trees + input prep; cues still Python"},
    {"item": "bad-signal handling", "V4": "window marked 'unknown' (gap > 0.5 s or > 25 % missing)",
     "V5": "window skipped (> 20 % missing); gaps <= 60 ms filled"},
    {"item": "presence model", "V4": "yes, separate (4 presence datasets)", "V5": "no (count 0 = empty)"},
])
DEV

In [ ]:
# 9. Figures + BENCHMARK.md
def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```" + chr(10) + df.to_string(index=False) + chr(10) + "```"


MAIN.to_csv(os.path.join(OUT, "main_scores.csv"), index=False)
NOISE.to_csv(os.path.join(OUT, "noise_scores.csv"), index=False)
DEV.to_csv(os.path.join(OUT, "device.csv"), index=False)
ALL.to_csv(os.path.join(OUT, "window_predictions.csv"), index=False)

# ---- main scores ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["exact", "mae", "f1_4"]):
    pv = MAIN.pivot_table(index="protocol", columns="system", values=metric, observed=True)
    pv.plot.bar(ax=ax, rot=0)
    ax.set_title(metric)
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "figures", "main_scores.png"), dpi=120)
plt.show()

# ---- noise ----
lab = NOISE["noise"] + " " + NOISE["level"].astype(str)
fig, ax = plt.subplots(figsize=(14, 4.5))
for system, g in NOISE.assign(cond=lab).groupby("system", observed=True):
    ax.plot(g["cond"], g["exact"], "o-", label=system)
ax.set_ylabel("exact (common span)")
ax.set_title("B3 - exact count under V4's noise suite")
ax.tick_params(axis="x", rotation=70)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT, "figures", "noise_exact.png"), dpi=120)
plt.show()

# ---- winners per benchmark ----
win_rows = []
for proto, g in MAIN.groupby("protocol"):
    for metric, better in [("exact", max), ("mae", min), ("f1_4", max), ("file_exact", max), ("recall_3", max)]:
        vals = g.set_index("system")[metric].dropna()
        if len(vals):
            best = better(vals)
            win_rows.append({"benchmark": proto, "metric": metric,
                             "best": ", ".join(str(s) for s, v in vals.items() if abs(v - best) < 1e-9),
                             **{str(s): round(float(v), 3) for s, v in vals.items()}})
ns = noise_summary.reset_index()
for metric, better in [("exact", max), ("mae", min), ("answered", max)]:
    vals = ns.set_index("system")[metric]
    best = better(vals)
    win_rows.append({"benchmark": "B3_noise (mean of 22)", "metric": metric,
                     "best": ", ".join(str(s) for s, v in vals.items() if abs(v - best) < 1e-9),
                     **{str(s): round(float(v), 3) for s, v in vals.items()}})
WIN = pd.DataFrame(win_rows)
WIN.to_csv(os.path.join(OUT, "winners.csv"), index=False)
display(WIN)

show = ["protocol", "system", "windows", "exact", "mae", "within1", "recall_0", "recall_1", "recall_2", "recall_3",
        "empty_false_alarm", "balanced_acc", "f1_4", "file_exact"]
lines = [
    "# V4 vs V5 - people-counting benchmark (ESP32-S3, WiCount)", "",
    f"Generated {time.strftime('%Y-%m-%d %H:%M')} on Kaggle. V4 = GitHub Nithin031/Vibe-Coded-Pipeline branch v4, "
    f"commit `{V4_COMMIT}`, run with its own code. V5 = esp32_counter.py.", "",
    f"Harness check: V4 LightGBM on its own protocol and windows = exact {score(v4_check)['exact']:.3f} "
    "(published 0.84).", "",
    "## Winners", "", md(WIN), "",
    "## B1 (V4 protocol) and B2 (new session, leave one file out)", "",
    md(MAIN[show].round(3)), "",
    "## B3 - noise (mean over 22 conditions)", "",
    md(noise_summary.reset_index().round(3)), "",
    "### Exact count per noise condition", "",
    md(NOISE.pivot_table(index=["noise", "level"], columns="system", values="exact", observed=True)
       .round(3).reset_index()), "",
    "## B4 - device", "", md(DEV), "",
    "## How to read this", "",
    "- Scored windows: window end >= 15 s and outside the first 30 s of the empty file, for every system.",
    "- Each system uses its own display smoothing (V4 ~5 s, V5 30 s); a longer median is steadier but slower.",
    "- B1 tests counts 0 and 3 on time blocks of the SAME recording the other blocks trained on (easy for those counts).",
    "- B2 holds out whole files: an ESP32-only model has no 0- or 3-person training file when that file is tested.",
    "- 'answered' = share of the clean scored windows that still get an answer under noise "
    "(V4 says 'unknown', V5 skips the window).",
    "- V5 final also trains on mat80 + WiMANS (other rooms, other hardware); V5 WiCount-only uses exactly V4's data.",
    "- Not compared: presence on the V4 presence datasets (V5 has no presence model), V4's WiCount -> OTH cross-room "
    "test (the OTH files are not available).",
]
with open(os.path.join(OUT, "BENCHMARK.md"), "w") as f:
    f.write(chr(10).join(lines))
print("saved", os.path.join(OUT, "BENCHMARK.md"))

In [ ]:
# 10. Keep the output small: drop the cloned repo, V4 cache and the symlinked data
import shutil

for path in [REPO, V4DATA]:
    shutil.rmtree(path, ignore_errors=True)
for name in sorted(os.listdir(OUT)):
    print(" ", name)